In [ ]:
! pip install evaluate

In [1]:
import numpy as np

In [2]:
from transformers import pipeline

In [5]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments, Trainer

from datasets import load_dataset
import evaluate

## Fine-tuning a pre-trained model (Trainer API)
Fine tuning a BERT model for text classification

In [6]:
# Pre-trained model to be used
checkpoint = 'bert-base-uncased'

In [7]:
# Tokenize the training data
# Dataset: Microsoft Research Paraphrase Corpus (MRPC) dataset, Link to paper: https://aclanthology.org/I05-5002.pdf)
# The dataset contains 5801 sentence pairs each labelled manually if one is a paraphrase of the other

raw_datasets = load_dataset('glue', 'mrpc')
raw_datasets

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

mrpc/test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [6]:
# Preview a pair in the dataset
raw_datasets["train"][11]

{'sentence1': 'The Nasdaq composite index increased 10.73 , or 0.7 percent , to 1,514.77 .',
 'sentence2': 'The Nasdaq Composite index , full of technology stocks , was lately up around 18 points .',
 'label': 0,
 'idx': 12}

In [7]:
# Check dataset features/labels
raw_datasets['train'].features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

### Data preprocessing

In [8]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
# Function to aid tokenizing (since the dataset is within a dictionary)
def tokenize_function(example):
    """
    Input: takes a dictionary (items of the dataset)
    Output: a dictionary (with keys sentence 1/2, label, ids, attn mask etc.)
    """
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

In [10]:
# Apply tokenization to the dataset
tokenized_dataset = raw_datasets.map(tokenize_function, batched = True)

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

In [10]:
# inspect
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [11]:
# Dynamic padding
# We apply padding within each individual batch (more efficient as we only pad according to the lonest sequence within that particular batch)
# To be used later in the training step

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

### Training & Evaluation

In [12]:
# Evaluation function (to help return metrics during training process)

def compute_metrics(eval_preds):
    """
    Args
        eval_preds : predictions (logits) from the model
    """
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis = -1) # Convert logits to binary predictions

    return metric.compute(predictions = predictions, references = labels)

In [ ]:
# Define training arguments
training_args = TrainingArguments(output_dir = "test-trainer", 
                                  eval_strategy = "epoch",
                                  fp16 = True, # enables mixed-precision for faster training and reduced memory usage
                                  )

# Define the model
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels = 2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
# Define the trainer
trainer = Trainer(
    model,
    training_args,
    train_dataset = tokenized_dataset["train"],
    eval_dataset = tokenized_dataset["validation"],
    data_collator = data_collator,
    processing_class = tokenizer,
    compute_metrics = compute_metrics
)

In [15]:
# Train and evaluate
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.455743,0.803922,0.873016
2,0.536719,0.417402,0.848039,0.892361
3,0.327971,0.705699,0.850490,0.898164


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1377, training_loss=0.3608114482184294, metrics={'train_runtime': 130.1133, 'train_samples_per_second': 84.572, 'train_steps_per_second': 10.583, 'total_flos': 405114969714960.0, 'train_loss': 0.3608114482184294, 'epoch': 3.0})

## Fine-tuning a pre-trained model (from scratch with PyTorch)

In [ ]:
## TBD later

## Fine-tuning an LLM with LoRA
Fine turning a model to generate Python code

Reference: https://abvijaykumar.medium.com/fine-tuning-llm-parameter-efficient-fine-tuning-peft-lora-qlora-part-2-d8e23877ac6f

In [28]:
!pip install -q -U trl transformers accelerate #git+https://github.com/huggingface/peft.git
!pip install -q datasets bitsandbytes einops 
!pip install -q wandb

In [29]:
from datasets import load_dataset
from random import randrange

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, AutoPeftModelForCausalLM # for LoRA

from trl import SFTTrainer, SFTConfig # Transformer Reinforcement Learning package (for fine-tuning)

from huggingface_hub import login, notebook_login

import wandb # For evaluation

In [30]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

In [31]:
# Define Model, Dataset
model_name = "Salesforce/codegen-350M-mono"
dataset_name = "iamtarun/python_code_instructions_18k_alpaca"
device_map = {"":0}

# W&B project name
wnb_project_name = 'python-fine-tuning'

In [32]:
# Define LoRA configuration
lora_config = LoraConfig(
    r = 64, # rank
    lora_alpha = 16, # scaling factor for the weight matrices
    lora_dropout = 0.1, # each neuron has a dropout probability of 10% to prevent overfitting
    bias = "none",
    task_type = "CAUSAL_LM"
)

In [33]:
# Define QLoRA configuration
qlora_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16"
)

In [34]:
load_dotenv()

# HuggingFace login
#login(token=os.getenv('HF_TOKEN'))

# W&B login
wandb.login(key=os.getenv('WANDB_API_KEY'))

True

In [ ]:
#load_dotenv() 
#wandb.init()

In [66]:
# Load the instruction dataset
dataset = load_dataset(dataset_name, split = 'train')

In [67]:
# Preview the instruction dataset
print(dataset)
print(dataset[5])

Dataset({
    features: ['instruction', 'input', 'output', 'prompt'],
    num_rows: 18612
})
{'instruction': 'Write a Python code to get the third largest element in a given row.', 'input': '[12, 13, 13, 45, 22, 99]', 'output': 'def third_largest(lst):\n    if len(lst) < 3:\n        return\n    distinct = []\n    for i in lst:\n        if i not in distinct:\n            distinct.append(i)\n    distinct.sort(reverse=True)\n    return distinct[2]', 'prompt': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nWrite a Python code to get the third largest element in a given row.\n\n### Input:\n[12, 13, 13, 45, 22, 99]\n\n### Output:\ndef third_largest(lst):\n    if len(lst) < 3:\n        return\n    distinct = []\n    for i in lst:\n        if i not in distinct:\n            distinct.append(i)\n    distinct.sort(reverse=True)\n    return distinct[2]'}


In [80]:
# Function to get a single text string with instruction, input and output from each row in the dataset
def prompt_instruction_format(sample):
  
  prompt =  f"""### Instruction:
  Use the Task below and the Input given to write the Response, which is a programming code that can solve the following Task:

  ### Task:
  {sample['instruction']}
  
  ### Input:
  {sample['input']}

  ### Response:
  {sample['output']}
  """

  return prompt

In [68]:
# Appy the function to add a 'text' column to the dataset with the formatted prompt for each row
dataset = dataset.map(lambda sample: {"text": prompt_instruction_format(sample)})
print(dataset)

Map:   0%|          | 0/18612 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output', 'prompt', 'text'],
    num_rows: 18612
})


In [39]:
# Load the pre-trained codegen-350M model
model = AutoModelForCausalLM.from_pretrained(model_name, 
          quantization_config=qlora_config,  # pass QLora parms
          use_cache = False, 
          device_map=device_map)
model.config.pretraining_tp = 1

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-mono
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [42]:
# Define the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [81]:
# Define training arguments to fine-tune the model
trainingArgs = SFTConfig(
    output_dir='./finetuned-models',
    #dataset_text_field='text',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True, # reduce memory consumption (re-computes some intermediate activations instead of storing them all)
    optim="paged_adamw_32bit",
    logging_steps=5,
    save_strategy="epoch",
    learning_rate=2e-4,
    weight_decay=0.001, # Regularization - adds a term to the loss fn. to penalize large weights
    max_grad_norm=0.3,
    warmup_steps=0.03, # influences learning rate
    lr_scheduler_type="cosine",
    disable_tqdm=True,
    report_to="wandb", # To report to Weights & Biases
    seed=42,
    completion_only_loss = False,
    max_length=2048,
    packing = False
)

In [82]:
# Create the trainer
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    peft_config = lora_config,
    formatting_func = prompt_instruction_format, # Note: Leave out in case of using a created "text" field in the dataset
    args = trainingArgs,
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:301: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Applying formatting function to train dataset:   0%|          | 0/18612 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/18612 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/18612 [00:00<?, ? examples/s]

KeyError: 'completion'

In [61]:
trainer.train() 

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 50256}.


{'loss': '1.204', 'grad_norm': '0.03491', 'learning_rate': '3.81e-06', 'entropy': '1.265', 'num_tokens': '1.519e+04', 'mean_token_accuracy': '0.7537', 'epoch': '0.002149'}


KeyboardInterrupt: 

In [79]:


import inspect
import trl
print(trl.__version__)
from trl import SFTTrainer
params = inspect.signature(SFTTrainer.__init__).parameters
for name, param in params.items():
    print(f"{name}: {param.default}")

0.29.0
self: <class 'inspect._empty'>
model: <class 'inspect._empty'>
args: None
data_collator: None
train_dataset: None
eval_dataset: None
processing_class: None
compute_loss_func: None
compute_metrics: None
callbacks: None
optimizers: (None, None)
optimizer_cls_and_kwargs: None
preprocess_logits_for_metrics: None
peft_config: None
formatting_func: None
